# trainer-class-skeleton — worked example 1: Trainer with global step counter and per-epoch loss logging

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `trainer-class-skeleton`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

A well-structured Trainer class separates concerns across lifecycle methods: `__init__` wires everything up, `_step` handles the forward-loss computation, `fit` drives the epoch loop, and `validate` evaluates on held-out data. Maintaining a `step` counter alongside a `history` dict gives you a training transcript that supports plotting, early stopping, and checkpointing without modifying the core loop.

## Worked solution

**Step 1 – `__init__`.** We store `model`, `optimizer`, `train_loader`, `val_loader`, `loss_fn` as attributes. We also initialize `self.step = 0` (counts gradient updates) and `self.history = {'train_loss': [], 'val_loss': []}`. Setting up the history here means any caller can inspect it after `fit` returns.

**Step 2 – `_step(x, y)`.** This method computes the loss only — it does NOT call `backward`. The forward pass (`logits = self.model(x)`) and loss (`self.loss_fn(logits, y)`) happen here. Keeping backward out of `_step` means subclasses can override `_step` (e.g. to log additional metrics) without worrying about the backward pass.

**Step 3 – `fit(n_epochs)`.** For each epoch we call `self.model.train()` then iterate the train loader. For each batch: `_step` → `backward` → `optimizer.step()` → `optimizer.zero_grad()` → increment `self.step` → append loss to history. After all batches, call `self.validate()`.

**Step 4 – `validate()`.** Switch to `model.eval()`, accumulate loss under `t.inference_mode()` weighted by batch size (so unequal final batches don't skew the average), append the mean to history.

In [ ]:
import torch as t
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

class WorkedTrainer1:
    def __init__(self, model, optimizer, train_loader, val_loader, loss_fn):
        self.model = model
        self.optimizer = optimizer
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.loss_fn = loss_fn
        self.step = 0
        self.history = {'train_loss': [], 'val_loss': []}

    def _step(self, x, y):
        logits = self.model(x)
        return self.loss_fn(logits, y)

    def fit(self, n_epochs):
        for _ in range(n_epochs):
            self.model.train()
            for x, y in self.train_loader:
                loss = self._step(x, y)
                loss.backward()
                self.optimizer.step()
                self.optimizer.zero_grad()
                self.step += 1
                self.history['train_loss'].append(loss.item())
            self.validate()

    def validate(self):
        self.model.eval()
        total, count = 0.0, 0
        with t.inference_mode():
            for x, y in self.val_loader:
                loss = self.loss_fn(self.model(x), y)
                total += loss.item() * x.shape[0]
                count += x.shape[0]
        self.history['val_loss'].append(total / count)

# Demo
t.manual_seed(0)
X = t.randn(40, 4)
Y = X @ t.tensor([1., -1., 0.5, 2.]) + 0.1 * t.randn(40)
train_ds = TensorDataset(X[:30], Y[:30])
val_ds = TensorDataset(X[30:], Y[30:])
train_dl = DataLoader(train_ds, batch_size=10)
val_dl = DataLoader(val_ds, batch_size=10)
model = nn.Linear(4, 1)
opt = t.optim.SGD(model.parameters(), lr=0.05)
trainer = WorkedTrainer1(model, opt, train_dl, val_dl, nn.MSELoss())
trainer.fit(3)
print('steps:', trainer.step)
print('val losses:', trainer.history['val_loss'])